# SHIPP — notebook bootstrap (do not run directly)

Every stage notebook starts with:

```python
%run ../common/shipp_silver_lib
```

This cell finds the repo root, **reloads** the Python modules under `config/` and `data_pipeline/`
(so an edited module is always picked up, even without *Clear state*), and exposes their functions
with the notebook's `spark` session already bound.

**All logic lives in `.py` files** (`data_pipeline/…`), covered by `tests/unit` in CI.
Notebooks only read, call those functions, run gates and write.

In [ ]:
import importlib
import json
import os
import sys
import time
from functools import partial


def _find_repo_root(start: str) -> str:
    path = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(path, "data_pipeline")) and os.path.isdir(os.path.join(path, "config")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise RuntimeError(f"Could not find the repo root (config/ + data_pipeline/) above {start}")
        path = parent


REPO_ROOT = _find_repo_root(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Reload in dependency order so edits to .py files always take effect.
for _name in [
    "config.settings", "config.tables",
    "data_pipeline.common.geo", "data_pipeline.common.spark_io",
    "data_pipeline.quality.data_quality",
    "data_pipeline.silver.current_state", "data_pipeline.silver.pairs", "data_pipeline.silver.routes",
    "data_pipeline.gold.scoring", "data_pipeline.ingestion.ors_client",
]:
    importlib.reload(importlib.import_module(_name))

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from config.settings import *          # statuses, ORS settings, SCORING, ...
from config.tables import *            # CATALOG, TABLES and every table constant
from data_pipeline.common import spark_io as _io
from data_pipeline.common.geo import coord_key, haversine_km, now_ntz
from data_pipeline.quality.data_quality import (
    duplicate_key_count, null_count, out_of_range_coordinate_count, run_quality_gate,
)
from data_pipeline.silver import current_state as _cs
from data_pipeline.silver.current_state import (
    CURRENT_STATE_CHANGE_TYPES, SILVER_SOURCE_CHANGE_TYPES,
    cdc_newest_first, independent_current_keys, normalize_code, reconstruct_current_state,
)

# I/O helpers with this notebook's SparkSession bound
table_exists = partial(_io.table_exists, spark)
read_bronze_pinned = partial(_io.read_bronze_pinned, spark)
write_overwrite = partial(_io.write_overwrite, spark)
write_silver = write_overwrite
assert_exact_schema = partial(_io.assert_exact_schema, spark)
logical_snapshot = partial(_io.logical_snapshot, spark)
assert_silver_matches_bronze = partial(_cs.assert_silver_matches_bronze, spark)

print(f"SHIPP bootstrap loaded from {REPO_ROOT}")